### Introduction to Audio and Music Processing (CH-EAM-B)
# 07 - Time-Frequency-Transformations 2

Prof. Dr. Jakob Abeßer (jakob.abesser@uni-bamberg.de)

Last update: 08.06.2026

## Learning Objectives

- Mel Spectrogram
  - Visual description of different sounds
  - Number of Mel bands
  - Compression
  
- Constant-Q Transform (CQT)
  - resolution for stable / time-varying pitches

## Audio Material

We are using two audio sources today:

1. Two randomly-mixed synthetic soundscapes from the Urban Sound Monitoring (USM) dataset (https://github.com/jakobabesser/usm):
- **339_mix.wav** - speech (male, female child) + running enging 
- **445_mix.wav** - drill, water drops, dog barking

2. Musical content
- **cmaj_3_instr.wav** - Arpeggios of C-major chord played by piano, xylophone, guitar, and synthesizer
- **262274__the_sound_side__abide-with-me-opera-vocals_short.wav** Opera singing recording from Freesound.org

In [ ]:
!pip install wget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import wget
import zipfile
import os
import glob

import librosa
import librosa.display
import soundfile as sf


from pathlib import Path
from IPython.display import Audio, display, Markdown

In [ ]:
# download zip file (if it has been downloaded already)
if not os.path.isfile('CH-EAM-B-Seminar-06_sounds.zip'):
    print('Please wait a couple of seconds ...')
    wget.download('https://github.com/CHBamberg/CH-EAM-B-SS-2026/raw/refs/heads/main/data/CH-EAM-B-Seminar-06_sounds.zip', 
                      out='CH-EAM-B-Seminar-06_sounds.zip', bar=None)
    print('CH-EAM-B-Seminar-06_sounds.zip downloaded successfully ...')
else:
    print('Files already exist!')

# if at least one of the audio files does not exist -> unzip zip file
if not os.path.isfile('cmaj_3_instr.wav'):
    print("Let's unzip the file ... ")
    assert os.path.isfile('CH-EAM-B-Seminar-06_sounds.zip')
    with zipfile.ZipFile('CH-EAM-B-Seminar-06_sounds.zip', 'r') as f:
        f.extractall('.')
        
    print("All done :)")
else:
    print('All audio files exist.')
    
dir_wav = 'CH-EAM-B-Seminar-06_sounds'

## Helper Functions

In [ ]:
def show_audio(y, sr, label=None):
    """ Show playback button to listen to audio file
    Args:
        y (np.ndarray): Audio samples
        sr (float): Sample rate (in Hz)
        label (str): Optional label
    """
    if label is not None:
        display(Markdown(f"**{label}**"))
    display(Audio(y, rate=sr, normalize=False))
    
def plot_waveform(y, sr, start_s=0.0, end_s=None, title="Waveform"):
    """ Plot waveform of audio recording or a segment thereof
    Args:
        y (np.ndarray): Audio samples
        sr (float): Sample rate (in Hz)
        start_s (float): Segement start (in s)
        end_s (float): Segment end (in s), if None: end of the file is used
        title (str): optional figure title
    """
    start = int(start_s * sr)
    end = len(y) if end_s is None else int(end_s * sr)
    end = min(end, len(y))
    start = max(0, start)

    t = np.arange(start, end) / sr

    plt.figure(figsize=(10, 2.5))
    plt.plot(t, y[start:end], linewidth=1)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# Let's listen to the soundscape audio files
fn_list = ('339_mix.wav',
           '445_mix.wav')
for fn_wav in fn_list:
    x, fs = librosa.load(os.path.join(dir_wav, fn_wav), sr=None)
    plot_waveform(x, fs, title=os.path.basename(fn_wav))
    show_audio(x, fs)

In [ ]:
## Parameters
HOP_LEN = 512
N_FFT = 2048

## 1) Mel spectrogram

Let us first study **339_mix.wav**.

In [ ]:
# TASK 1a) Create an STFT magnitude spectrogram with logarithmic magnitude scaling and plot it
fn_wav = os.path.join(dir_wav, '339_mix.wav')
x, sr = librosa.load(fn_wav, sr=None)    
# TODO add stft call with N_FFT and HOP_LEN
X     = None 
# TODO add dB scaling
Xm    = None 
fig, ax = plt.subplots(figsize=(10,4))  
librosa.display.specshow(Xm,
                         sr=sr,
                         hop_length=HOP_LEN,
                         n_fft=N_FFT,
                         x_axis="time", y_axis="hz",
                         ax=ax,                 
                         cmap="magma",
                         vmin=-80, vmax=0)
plt.colorbar(ax.collections[0], ax=ax).set_label("dB")
plt.show()

**Observations**

The main two sound sources are **human voice** (male and female kid) and a **running engine** sound.
Can you spot both?


In [ ]:
# Task 1b) repeat for "445_mix.wav", can you spot the drill, water drops, and dog barking sound, 
#          describe them visually in the spectrogram!



### Compression with Mel spectrogram

Now we want to compute a Mel spectrogram and find a suitable **number of Mel bands** that captures all details we need while **reducing the overall number of time-frequency bins (compression)**. For this purpose, we visually compare the Mel spectrogram with the STFT magnitude spectrogram to compare the "level of detail".

In [ ]:
# Compute the Mel spectrogram for different number of Mel bands: {16, 32, 64, 128, 256}

# load audio file
fn_wav = os.path.join(dir_wav, '339_mix.wav')

# parameters
N_MELS = [16, 32, 64, 128, 256]

# compute Mel spectrograms with different number of Mel bands (e.g. use a list comprehension)
mel_specs = [
    librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=sr,
                                       n_fft=N_FFT, hop_length=HOP_LEN,
                                       n_mels=n),
        ref=np.max
    )
    for n in N_MELS
]

# prepare figure
n_rows = len(N_MELS)
fig = plt.figure(figsize=(8, 2 * n_rows))

for i in range(len(N_MELS)):
    # plot one Mel spectrogram per row
    ax  = plt.subplot(len(N_MELS), 1, i+1)
    
    librosa.display.specshow(mel_specs[i], sr=sr, hop_length=HOP_LEN,
                             x_axis="time", y_axis="mel",
                             fmin=0, fmax=sr // 2,
                             ax=ax, cmap="magma",
                             vmin=-80, vmax=0)

    
    ax.set_ylabel("Hz", fontsize=8)
    ax.set_title(f"{N_MELS[i]} Mel bands", fontsize=9)
    cb = fig.colorbar(img, ax=ax, pad=0.01, fraction=0.02)
    cb.set_label("dB", fontsize=7); cb.ax.tick_params(labelsize=7)
    ax.set_xlabel("Time (s)")

plt.tight_layout()
# plt.savefig("spectrogram_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation**: What is a good choice of the number of Mel bands with a sufficient resolution in comparison to the STFT spectrogram? Explain why?

In [ ]:
# Task 1c) Compute the compression ratio (size of Mel spectrogram / size of STFT spectrogram)
HOP_LEN = 512
N_FFT = 1024
fn_wav = os.path.join(dir_wav, '339_mix.wav')
x, sr = librosa.load(fn_wav, sr=None)          # ← store in x
X     = librosa.stft(x, n_fft=N_FFT,           # ← use x, not y
                     hop_length=HOP_LEN)

# again for Mel spec (assuming 128 Mel bands)
Xm = librosa.feature.melspectrogram(y=x, sr=sr,
                                       n_fft=N_FFT, hop_length=HOP_LEN,
                                       n_mels=128)

# TODO compression ratio
compression_ratio = None 
print(f"Compression ratio: Mel spectrogram takes {(compression_ratio):2.2f}% of the original size.")

## 2. Constant-Q Transform (CQT)

In [ ]:
# Let's listen to the music files
fn_list = ('cmaj_3_instr.wav',
           '262274__the_sound_side__abide-with-me-opera-vocals_short.wav')
for fn_wav in fn_list:
    x, fs = librosa.load(os.path.join(dir_wav, fn_wav), sr=None)
    plot_waveform(x, fs, title=os.path.basename(fn_wav))
    show_audio(x, fs)

We first want to compare the STFT and the CQT.

In [ ]:
# Task 2a) Plot the STFT magnitude spectrogram with dB scaling for 'cmaj_3_instr.wav':

Why do you think is it hard to estimate the fundamental frequency values?

In [ ]:
# Task 2b: Plot a CQT with
#            - frequency resolution of 12 bins per octave (1 bin per semitone)
#            - minimum frequency of 27.5 Hz (A0, lowest key on the piano)
#            - range of 7 octaves
fn_wav = os.path.join(dir_wav, 'cmaj_3_instr.wav')
x, sr = librosa.load(fn_wav, sr=None)          

# CQT parameters 
BINS_PER_OCTAVE = 12       
N_OCTAVES = 7                           
N_BINS = BINS_PER_OCTAVE * N_OCTAVES
FMIN = librosa.note_to_hz("A0") 

# TODO compute CQT
C = None
Cdb = librosa.amplitude_to_db(np.abs(C), ref=np.max)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,4))  
img = librosa.display.specshow(Cdb, sr=sr, hop_length=HOP_LEN,
                                x_axis="time", y_axis="cqt_note",
                                fmin=FMIN, bins_per_octave=BINS_PER_OCTAVE,
                                ax=ax, cmap="magma", vmin=-80, vmax=0)
cb = fig.colorbar(img, ax=ax, pad=0.01, fraction=0.02)
cb.set_label("dB", fontsize=8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Pitch")
plt.tight_layout()
plt.show()

**Observation**: What changed? How does the spectral pattern (fundamental frequency + overtone frequencies) change for increasing pitches (is it stretched or just shifted?)

### Frequency resolution

Now let us examine the opera singing example. Here, the pitch frequencies are not stable but constantly vary. For instance, we observe a periodic pitch modulation (vibrato), which is typical for this style of singing.

In [ ]:
# Task 2c) Create a similar CQT plot as before for "262274__the_sound_side__abide-with-me-opera-vocals_short.wav"

In [ ]:
# Task 2d) Redo but this time with 5 bins per semitone to have a finer spectral resolution

**Final observation**: For stable pitch frequencies, 12 bins per octave is a good choice. For time-varying pitch contours, use 3 or 5 bins per octave.

done :)